<h1 style="text-align:center; color:#2E86C1;">
Critical Scale Invariance in a Healthy Human Heart Rate
</h1>

<h3 style="text-align:center;">
Group Members
</h3>

<p style="text-align:center;">
Luca Di Turi<br>
Libero Pollini<br>
Alessandro Turino<br>
Mattia Ziglioli
</p>

# Abstract

In this notebook, we analyze the heartbeats of healthy individuals, reproducing the experimental analysis of Kiyono presented in paper [\[4\]](#ref4). We will investigate the probability distribution function of heart beat time intervals from the [Fantasia](https://physionet.org/content/fantasia/1.0.0/) dataset. The data will first be fitted using a Gaussian model, and then using the Castaing model inspired by high Reynolds number turbolence effects on fluid velocities. We will test the scale invariance of the distribution of interbeats times with respect to the temporal parameter $s$.
Finally, we will examine how the parameter $\lambda^2$ varies with the polynomial order $q$ used to fit the temporal differences between individual heartbeats.


# The heart: a complex system


A healthy human heart rate belongs to a special class
of **complex signals** including (but not limited to) genetics, physical fitness, stress or psychological status, diet, drugs, hormonal status, environment, and disease/illness, as well as the interaction between these factors. Our goal is to analyze it from a from a mathematical perspective.

We will mainly focus our attention to the probability distribution function of the cardiac interbeat times, defined as the time differences between consecutives heart contractions. As we will see, this PDF does not follow a Gaussian statistic [\[1\]](#ref1) , but presents a robust scale invariance. This suggests the idea that a healthy cardiac system operates near a critical and out of equilibrium state [\[2\]](#ref2).

The mechanism responsible for complex heart rate dynamics is not yet fully understood. Since these dynamics mirror autonomic control of heart rate and can help predict mortality in cardiac patients, clarifying this mechanism is essential.

## Sequential heart interbeat intervals (IBIs)

An Electrocardiogram (**ECG**) is a recording of the heart's electrical activity through repeated cardiac cycles. It is an electrogram of the heart which is a graph of voltage versus time of the electrical activity of the heart using electrodes placed on the skin.

In order to analyze the ECG data we have to use Heart Rate Variability metric. Sequential heart interbeat intervals (**IBIs**) are the varying time gaps between each heartbeat, which form a time series that can be analyzed. The standard to measure IBIs is the detection of the R-wave peak on an ECG, leading to **"RR intervals"**. 

<div style="display:flex; justify-content:center; gap:20px;">

  <div>
    <img src="images/EKG.jpg" width="500"><br>
    <b>Figure 1:</b> ECG Signal in an heart cycle.
  </div>

  <div>
    <img src="images/RRinterval.png" width="500"><br>
    <b>Figure 2:</b> RR Intervals plot.
  </div>

</div>

During each heartbeat, a healthy heart has an orderly progression of depolarization that starts with pacemaker cells in the sinoatrial node, spreads throughout the atrium, and passes through the atrioventricular node down into the bundle of His and into the Purkinje fibers, spreading down and to the left throughout the ventricles.

In the 12-derivation ECG, four electrodes are placed on the patient's limbs and six on the chest surface. Then the overall electrical potential of the heart is measured in twelve points ("derivations") and is recorded for a given period of time, typically ten seconds. In this way, the general amplitude and direction of the electrical depolarization of the heart is captured at all times and throughout the heart cycle.

# Datasets used

The data used to reproduce the analysis are taken from [PhysioNet](https://physionet.org/), a public database.

First we analyzed [Fantasia](https://physionet.org/content/fantasia/1.0.0/) dataset: it is a collection of recordings from 20 young (21-34 years old) and 20 elder (68 - 85 years old) individuals. All subject were selected after a strict health check, to verify the absence of any pathological rythms. The ECG signals were taken for a total of 120 minutes, sampled at 250 Hz, while each participant was watching the Disney movie Fantasia in order to help mantain wakefulness. Together with ECG signals, respiration and, only in some individuals, blood pressure were also recorded.

Plot the representing population of the dataset:

<div>
    <img src="images/age_distribution_Fantasia.png" width="1500"><br>
    <b>Figure 3:</b> Fantasia Dataset
</div>

Fantasia dataset contains recordings from 40 different subjects. The first 20 subject's age is ranges from 21 to 34 years old and the corresponding files are marked with the letter "y". The remaining subjects'age ranges from 68 to 86 years old and the letter "o" is used instead, as shown in Fig.3.

# Dataframe analysis 

Importing the libraries:

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import os 
import re
from IPython.display import Image,display
import utility as util  # python file with functions
import importlib
import warnings
from IPython.core.interactiveshell import InteractiveShell

## Fantasia dataframe

```python

wget -r -N -c -np https://physionet.org/files/fantasia/1.0.0/

```

The `.hea` files contain the metadata, where we can read all the basics information about the files. 

The `WFDB` Python package was used to access this type of data files. This open-source library is specifically designed to interface with PhysioNet datasets and offers robust tools for peak detection on raw ECG signals, along with convenient visualization utilities and a high degree of customization and configuration.

The raw ECG data instead was stored in `.dat` format. Blood pressure and respiratory data are here neglected and not taken into account on the following analysis. ECG data are stored as an ordered sequence of values of voltage differences, sampled at 250 Hz with a conversion factor of 2000 adu/mV (analog digital units).


```python

import wfdb


record_path = "./physionet.org/files/fantasia/1.0.0/f1o01"

record = wfdb.rdrecord(record_path)

# Sampling frequency
print("Sampling frequency:", record.fs)

# Channel names
print("Signal names:", record.sig_name)

# Units
print("Units:", record.units)

# Number of samples
print("Number of samples:", record.sig_len)

# Duration in seconds
print("Duration (s):", record.sig_len / record.fs)
```

Let's analyze this dataframe. We have 20 young (21 - 34 years old) whose files are indicated with a **y** as **"f1y01"**. Then we have 20 elderly (68 - 85 years old) whose files are indicated with a **o** as **"f1o01"**. Each subgroup of subjects includes equal numbers of men and women (see **figure 3**). They are rigorously-screened healthy subjects underwent 120 minutes of continuous supine resting while continuous electrocardiographic and respiration signals were collected. Half of those files in each group (the **f2*** records) include a blood pressure waveform. 

In the following pictures we made a comparison between a real ECG and the data how appear the data stored in Fantasia dataset:

<div style="display:flex; justify-content:center; gap:20px;">

  <div>
    <img src="images/QRS_complex.jpg" width="600"  style="margin-top: 70px;" ><br>
    <b>Figure 4:</b> QRS complex in a ECG
  </div>

  <div>
    <img src="images/eg_fantasia_data.png" width="600"><br>
    <b>Figure 5:</b> Plot of 4 heart pulses from Fantasia file f1o01
  </div>

</div>

Luckily for each patient there is a precomputed `.txt` annotation file , containing the sample indexes for witch the ECG signal peaks, i.e. at the R peaks. For the Fantasia analysis, this file was used to avoid the need for manual R-peak detection.

## LOAD the data

First we set a possible Path and Key for Database Fantasia: in this way it's easier if in future we want to add more dataframe 


In [ ]:
paths = {'Fantasia':'physionet.org/files/fantasia/1.0.0/subset/'}
keys= {'Fantasia':'RR_intervals_original_Fantasia'}
extentions={'Fantasia': 'dat'}


In [ ]:
database='Fantasia' 
path = paths[database]
key = keys[database]
extension=extentions[database]
files= np.loadtxt(path+'RECORDS', dtype='str') # RECORDS is a list containing files names

print("File names for different patients:\n", files)

```python

# Run if the package wfdb is installed

def import_ecg_file(record_path):
    """
    Extract ECG,RESP and BP (if occurs) time series from Fantasia dataset.
    """
    record = wfdb.rdrecord(record_path)

    num_signals = record.p_signal.shape[1]

    # distingue se ci sono 2 o 3 segnali
    if num_signals == 2:
        col_names = ['RESP', 'ECG']
    elif num_signals == 3:
        col_names = ['RESP', 'ECG', 'BP']

    df_signals = pd.DataFrame(record.p_signal, columns=col_names)
    return df_signals


```

In [ ]:
def import_data_txt(head_folder, files):
    folder_path = Path(head_folder)
    if not folder_path.exists():
        raise ValueError(f"Folder does not exist: {head_folder}")
    
    files_lower = [f.lower() for f in files]
    
    
    txt_files = [p for p in folder_path.iterdir()
                 if p.is_file() and p.suffix == ".txt" and p.stem.lower() in files_lower]
    
    if not txt_files:
        raise ValueError("No annotation .txt files found. Check folder path!")
    
    txt_files.sort(key=lambda p: files_lower.index(p.stem.lower()))
    
    data = {}
    for filepath in txt_files:
        subject = filepath.stem
        arr = np.loadtxt(filepath, dtype=float)
        data[subject] = pd.Series(arr)  # convert each array to Series

    df = pd.DataFrame(data)
    return df

folder = "./physionet.org/files/fantasia/1.0.0/subset/"
RR_Fantasia_interval = import_data_txt(folder,files)
RR_Fantasia_interval

As we can see from the NaNs, it looks like the files are not the same length, and that's understandable since the heart rate can be different from one to one.

## Selecting a datafile

We can now choose to select a single file and focus on that one. For this presentation we selected a young candidate **Y1**:

In [ ]:
possible_beat=RR_Fantasia_interval.columns
signal_extraction=possible_beat[5] ### Here we can choose what file we want from the txt
print("File selected ",signal_extraction)

We extract the **Y1** data

In [ ]:
data=RR_Fantasia_interval[signal_extraction]
data_clean=data.dropna() ## drop NAN values
data_clean=data_clean.to_numpy()
data_clean

Now the data $b(i)$ are ready to be analyzed 

We can also plot the respective time series in the `.hea` file:  

```python
file_name = "./physionet.org/files/fantasia/1.0.0/f1" + signal_extraction[0].lower()+"0"+signal_extraction[1]
time_series = import_ecg_file(file_name)
time_series.head()

# ECG 
plt.close()
conversion_factor = 2000  # ADU/mV
sampling_freq=250 # fs in Hertz
ecg = time_series.iloc[:]['ECG'].to_numpy()/conversion_factor
resp = time_series.iloc[:]['RESP'].to_numpy()/conversion_factor
samples=ecg.shape

plt.plot(ecg[0:4*sampling_freq],color='orange', label='b(i)')
if time_series.shape[1] == 3:
    bp = time_series.iloc[:]['BP'].to_numpy()/conversion_factor
    #plt.plot(bp)

plt.grid(linestyle='dotted')
plt.xlabel('samples')
plt.ylabel('Voltage (mv)')
plt.legend(loc='best')
plt.title('TIME SERIES of heart pulse from ECG '+"f1" + signal_extraction[0].lower()+"0"+signal_extraction[1])
plt.show()
print("ECG shape:",samples )
print("Respiration shape:", samples)
```

# ANALYSIS CODE

## Polynomial fit and fluctuations $\Delta_{s}B$ of detrended data 

In this first part, we compute the cumulative function $\Delta_{s}B$ starting from the sequential heart interbeat intervals $b(i)$. We investigate the PDF of heart rate increments at different time scales "**s**" (see **s_values** )
  

## SET GLOBAL PARAMETERs

Here we set the global parameters of the data analysis : 

In [ ]:
number_of_scale=12 #### number of different s scales
s_values = [2**i for i in range(2,number_of_scale)]
q_values=[2,3,4,5] ## orders of the polynome we want to fit for the detrended data

We first integrate the $b(i)$ , $B(m)=\sum^{m}_{j}b(j)$ and the resultant $B(m)$ is divided in segments using sliding window of size 2s.
Then we want to eliminate the nonstationarity of the data using a local detrending function. The function `np.polyfit` fit a polynomial $p[0] * x^{deg} + ... + p[deg]$ of degree **deg** to points (x, y). Returns a vector of coefficients that minimises the squared error in the order deg, deg-1, … 0.

In [ ]:
B = np.cumsum(data_clean)

In each segment the best $q$ th order polynomial is fit to the data. By this procedure the $(q-1)$-th order polynomial trends are eliminated and we analyze the hole PDF of $\Delta_{s}B(i)$. For a fixed scale "$s$" The differences $\Delta_s B(i) = B^{*}(i+s) - B^{*}(i)$ are obtained by sliding in time over the segments, where $B^{*}(i)$ is a deviation from the polynomial fit.


In [ ]:
dfs = {}   # dictionary to hold all dataframes

The functions used are listed below

In [ ]:
# x = beat number
# y = time from b(i) and b(i+1)
def poly_fit(x, y, order=3):
    # revert order of coeffs from x^0 to x^3
    coeff = np.polyfit(x, y, order)
    p = np.poly1d(coeff)
    return p(x)

def get_deviation(y, s,order):
    #array deviations
    deviations = np.zeros(len(y))

    x = np.arange(len(y))
    segment_length = 2*s
    # number of sliding segments
    
    for k in range(0, len(y), segment_length):
        # I take a segment and fit
        x_slice = x[k:k+segment_length]
        y_slice = y[k:k+segment_length]
        y_fitted = poly_fit(x_slice, y_slice, order)
        dev = y_slice-y_fitted
        
        deviations[k:k+segment_length] = dev
    return deviations

Here we compute $\Delta_{s}B(i)$. For the the rest of the analysis we selected the data fitted with `poly_order`=**3**. At the end we discuss the quality of this fit.

In [ ]:
poly_order=3

In [ ]:
# prepare the array
warnings.filterwarnings("ignore")
InteractiveShell.ast_node_interactivity = "all"
deltaB = [] # empty list, to be filled with s values
for i in q_values:
    df = pd.DataFrame()
    for s in s_values:
            z = get_deviation(B, s, i)
            df[s] = z
            if i==poly_order: ## select poly_order=3
                deltaB.append(z)
    dfs[i] = df   # store dataframe under key = polynomial order 

In [ ]:
dfs[3].head()

An example of $\Delta_s B(i)$  at a fixed s=8: 

In [ ]:
plt.plot(deltaB[2])
plt.xlabel("Beat number [i]")
plt.ylabel("Time interval $\Delta_s B(i)$ [s]")
plt.legend(["s=8"])
plt.grid()

# Fitting the distributions with two models:
Following the steps of Kiyono's paper [\[4\]](#ref4),  we fit the Gaussian and Castaing's model to our dataset, in order to test the possible presence of nonlinear
mechanisms in complex heart rate dynamics.

The goal of this section is to estimate the probability density function (PDF) of the "fluctuations" from the mean of the time intervals between one heart beat and the next one ($\Delta B _s$), at a fixed "scale" $s$, for a given ECG dataset of a single healthy human heart. Then, we fit the data with a Gaussian and a given non Gaussian PDFs.
The latter essentially exhibits "fat tails" as compared to a Gaussian distribution.
Note that the code for extrapolating the fit parameters in this latter case is also included in the next section (here we simply used the fit results). 

## Gaussian Fit

First we start with a Gaussian fit:

$$
\mathcal{N}(x ; \mu , \sigma)= \frac{1}{\sqrt{2\pi \sigma^{2}}} exp\Bigg({\frac{(x - \mu)^{2}}{2\sigma^{2}}}\Bigg)
$$

from the theory, we expect a nice accuracy at the peak of the distribution of $\Delta_s B(i)$, while the tails should exhibit non-gaussian features.

## Non-Gaussian fit: The Castaing's equation

In order to obtain an appropriate fit for the fat tails observed in the dataset, we tried fitting with the Castaing's equation:

$$
\tilde{P_s}(x) = \int P_L\Big(\frac{x}{\sigma}\Big)\frac{1}{\sigma}G_{s,L}(ln\sigma)d(ln\sigma)
$$

In this method, the PDF at scale s, $\tilde{P_s}(x)$, is considered as a mixture of gaussian with different $\sigma$ values. In particular:

- $ P_L\big(\frac{x}{\sigma}\big)\frac{1}{\sigma}$ is a standard gaussian, the limit distribution of the increments.
- $G_{s,L}(ln\sigma)$ is the distribution of the standard deviations, assuming it is a gaussian in $ln\sigma$ ($\sigma$ Log-Normal) as in the paper.

Since the fitted pdf is standardized, $E[\sigma^2]=1$ (1), defining $\xi = ln\sigma$ and assuming  $\xi\sim N(\mu,\lambda^2)$, from (1) we get $\mu = -\lambda^2$. 

So, we only have a free parameter $\lambda^2$, and $G_{s,L}$ assumes the form:

$$
G_{s,L}(ln\sigma) = \frac{1}{\sqrt{2\pi}\lambda}\text{exp}\Big(-\frac{(ln\sigma+\lambda^2)^2}{2\lambda^2}\Big)
$$


Let us start by choosing a fixed scale $s$ (the choice of the scale is irrelevant: this will be shown explicitly later). We standardize the increments before feeding them to all functions (we check that the gaussian fit gives mean 0 and standard deviation 1, and then simply use these values later). 
We also choose an appropriate binning for the first and second graph:

In [ ]:
sc=2**(poly_order)
print("Scale:", sc)
# resulting fit parameter (from later code):
l2=0.167

df=dfs[poly_order]
data=df[sc]
cincs= util.standardize_increments(df[sc].dropna().values)

# bin edges for each of the two following histograms:
bes=np.linspace(cincs.min(), cincs.max(), 21)
bes2=np.linspace(cincs.min(), cincs.max(), 180)

Then, we use the function we implemented to superimpose the Gaussian and Castaing fitted functions to the data, for the fixed scale, note that in the second graph the dataset is still organized in an histogram, with only the center of the bins desplayed, to visualize the underlying PDF.

In [ ]:
# Gaussian vs Castaing fit:
fig1,(ax1,ax2)=plt.subplots(1,2,figsize=(12,7))
ax1.grid()
util.histo_kde(increments=cincs, h_ax=ax1, bin_edges=bes, scale=sc, hist_color='lightblue', line_color='red')
m,s=util.gaussian_vs_cast_fit(increments=cincs, g_ax=ax2, bin_edges=bes2, scatter_color='black', scatter_marker='o', line_color='black',lambda2=l2,scale=sc)

Note that the graph on the right clearly shows how the Gaussian fit fails to account for "fat" tails observed.

At this point, we want to check if our visual intuition is correct by performing an appropriate quantitative test.
We therefore defined a function to evaluate the goodness of both fits. 
As we wanted to avoid a simple chi-square test in light of its dependency on the choice of bins (and, generally, little sensitivity to tails), we tried an appropriate built-in function of Scipy: the so-called Kolmogorov-Smirnov (KS) test.
The result of this test is the "KS statistics", which is essentially the maximum vertical distance between the empirical and the predicted cumulative distribution functions (CDFs).
(See the next section for a detailed explanation of Castaing's PDF and CDF, which we implemented with appropriate functions).

We visualize the result by plotting the empirical and the two predicted (Gaussian and Castaing's) CDFs:

In [ ]:
fig2, ax3 = plt.subplots(figsize=(8, 5))
util.ks_tests(cincs, l2, ax3, sc)

Visually, Castaing's CDF is a much better fit to the data.
The KS test confirms this: the maximum vertical distance (relative to the curve's height in each point) between the empirical and fitted Gaussian's CDFs is 2-3 times greater than that between the empirical and the fitted Castaing's.

In the next section, we tried to estimate the goodness of the fit using a more elaborate statistic, the so-called "Anderson-Darling" test.

## Evaluating the quality of the fit through the Anderson–Darling test

The obtained fit can be evaluated in a way independent from the binning chosen to fit the PDF: through the Anderson–Darling test. 

It starts by evaluating the difference between the theoretical CDF ($F(x)$), in this case obtained from the Castaing function, and the experimental one ($F_n(x)$), a simple step function going from 0 to 1 with steps of $1/n$ (where $n$ is the length of the dataset). This difference, for each value of $x$, is squared and weighted by $w(x)$. 

For the Anderson-Darling test the weight function is $w(x)=[F(x)(1-F(x))]^{-1}$; this weight is particularly useful in our situation, since it values the tails of the CDF more: 

$$\text{as } x\rightarrow -\infty, \ F(x)\rightarrow 0 \quad \text{and} \quad \text{as } x\rightarrow +\infty, \ (1-F(x))\rightarrow 0$$

This is important since the tails are the regions where the fit to a normal function fails, due to the "fat tails" observed, and we expect the Castaing function to better represent the data.

The Anderson-Darling coefficient is given by:

$$A^2 = n \int \frac{(F_n(x)-F(x))^2}{F(x)(1-F(x))}dF(x)$$

Through discretization of the integral, we get the result:

$$A^2 = -n-S \quad , \text{with} \quad S = \sum^n_{i=1}\frac{2i-1}{n}[\ln(F(Y_i))+\ln(1-F(Y_{n-i+1}))]$$

where $Y_i$ is the i-th sorted data point.

Obtaining a significant value for this parameter isn't straightforward: common distributions have tabulated reference values that can be compared to the results obtained to determine whether or not the fit is representative of the dataset. In our case, we don't have any tabulated values to refer to: we had to create the table.

### The Monte Carlo Approach

After having obtained an optimal value for the parameter $\hat{\lambda}^2$ and for the coefficient $A_{obs}^2$, we had to generate a statistic for $A^2$ to evaluate the p-value of our result. 

Starting from the Castaing distribution with parameter $\hat{\lambda}^2$, we generated N different datasets. For every dataset, we repeated the fit operation, and thus we obtained a set of optimal parameters ${\hat{\lambda}^2_i}$. For each $\hat{\lambda}^2_i$, we calculated an $A_i^2$. The p-value was then calculated as the ratio between the number of $A_i^2$ bigger than $A_{obs}^2$ and the total number of $A_i^2$.

The pain didn't end there: for big samples, the Anderson–Darling test turned out to be very sensitive to the number of datapoints in the dataset; with $n\sim10^4$, the test tracks small differences between the sample and the theoretical curve, thus giving big $A^2$ values and vanishing p-values. The solution to this problem was ***subsampling***.
After calculating $A_{obs}^2$, we randomly selected subsets of size $M \sim 10^2$ from the original dataset and repeated the Monte Carlo procedure described above. This approach yielded more meaningful p-values, reflecting the physical validity of the model rather than statistical fluctuations due to sample size.

### Professor notes about this method:

To avoid waste of data, we should have divided the original sample in different sub-samples and we should have calculated the observed $A^2$ coefficient in the subset as the mean of the different coefficients obtained. 

In [ ]:
results, lambdas, lambda_errors, a_s = util.get_res(dfs[poly_order])

## Testing the scale invariance of the process:

To test the scale invariance of  $\Delta B _s$, we started by normalizing the PDFs obtained from the dataset at different $s$ values.


After having obtained $\Delta B _s(i) = B^*(i+s)- B^*(i)$ , we calculated $\sigma_s$ for each dataset, and we computed the PDFs of $\Delta B _s$ from the histogram over their distribution. To normalize these distribution and check the scale invariance, we did the variable change 

$$
\Delta B _s \rightarrow \Delta B _s/\sigma_s
$$

while keeping the integral normalized to one $y = P ( \Delta B _s ) \rightarrow y' = P ( \Delta B _s ) \cdot \sigma_s $ since :

$$
\int^{+\infty}_{-\infty}P(\Delta B_s)\ dB_s = 1 = \int^{+\infty}_{-\infty} ( P(\Delta B_s)\cdot \sigma_s )\  dB_s/\sigma_s
$$


In [ ]:
def normalize_ds(data):
    sigma = data.std()
    nbins = int(np.sqrt(len(data)))
    pdf, bin_edges = np.histogram(data, bins=nbins, density=True)
    

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    x = bin_centers / sigma

    y = pdf * sigma
      
    return(x,y)

def standardize_ds(data):
    sigma = data.std()
    nbins = int(np.sqrt(len(data)))
    pdf, bin_edges = np.histogram(data, bins=nbins, density=True)
    

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    x = bin_centers / sigma

    y = pdf 

    return (x, y)


def plot_collapse(increments_df, standardize=False):
    """
    increments_df è un dataframe dove le colonne sono i diversi valori di s
    """
    plt.figure(figsize=(10, 6))
    
    def plot_single_column(col):
        s = col.name
       
        # Rimozione eventuali NaN (non dovrebbero essercene)
        data = col.dropna()

        if standardize: 
            x, y = standardize_ds(data)
        else:
            x,y = normalize_ds(data)
        # Plot diretto
        plt.semilogy( x , y , ls='', marker = 'o',label=f's={s}',ms = 3)

    # Applichiamo la funzione a ogni colonna (axis=0 è il default)
    increments_df.apply(plot_single_column)

    if standardize:
        plt.title('Data Collapse: Standardized PDFs')
        plt.xlabel(r'$\Delta_s B / \sigma_s$')
        plt.ylabel(r'$P(\Delta_s B)$')
        plt.legend()
        plt.grid(True, which="both", ls="-", alpha=0.5)
        plt.show()
    else:
        plt.title('Data Collapse: Rescaled PDFs')
        plt.xlabel(r'$\Delta_s B / \sigma_s$')
        plt.ylabel(r'$\sigma_s \cdot P(\Delta_s B)$')
        plt.legend()
        plt.grid(True, which="both", ls="-", alpha=0.5)
        plt.show()

In [ ]:
plot_collapse(dfs[poly_order],standardize=True)

Now we check for the scaling invariance plotting the rescaled PDFs:

In [ ]:
plot_collapse(dfs[poly_order],standardize=False)

We can also apply a fit with the Castaign's model to the collapsed data:

In [ ]:
util.plot_results(results,lambdas,lambda_errors)

We expect that for sufficiently large $s$ ($s>2$), $\lambda^2$ will be constant. To assess our analysis, we fitted the different $\lambda^2$  values vs log($s$) to a linear function:

In [ ]:
util.lambda_comp( lambdas,lambda_errors,list(results) )

As expected, the slope of the line is compatible with zero. However, for large scales ($s>1024$), $\lambda^2$ tends to zero because the window size becomes too large relative to the dataset. When $\lambda$ approaches zero, the Castaing function converges to a Gaussian distribution, losing its characteristic fat tails. Furthermore, at high scales, the limited size of the dataset ($N$) becomes an issue: as $s$ approaches $N$, we obtain fewer $\Delta B_s​(i)$ samples, making the analysis less statistically significant

# $\lambda^2$ dependence on $s$ parameter

According to Kiyono [\[4\]](#ref4)., the fitting parameter $\lambda^2$ in Castaing’s equation depends on the order of the detrending polynomials. Li [\[4\]](#ref4). argues that, within the turbulent cascade framework applicable here, $\lambda^2$ can be interpreted as being proportional to the number of cascade steps and is therefore expected to decrease linearly with $\log(s)$. Our results show a decreasing trend consistent with this expectation, although the statistical significance is weak.

This can be seen as a further proof of the robust scale invariance hypothesis.

To justify the use of a third-degree polynomial for detrending, we compare results obtained with different polynomial orders. Except for the second-degree polynomial, the dependence of $\lambda^2$ on $s$ remains similar across all tested orders. Therefore, we perform the above analysis using a third-degree polynomial detrending.

```python
### code for lambda^2 vs s plot
s_values = [2**i for i in range(3,12)] # s goes from 8 to 2048
poly_order=[2,3,4,5] ## orders of the polynom we want to fit for the detrended data
```

<img src="images/lambda_2_vs_s.png" width="1000"><br>
**Figure 6:** $\lambda^2$ vs $\log(s)$.

As above, we report the results of the linear fit of $\lambda^2$ versus $\log(s)$, here shown for the different polynomial orders $q$.

In [ ]:
def lambda_comp(lambdas,lambda_errors,s,i):
    '''
        Evaluate the compatibility of the obtained lambda^2 values with a zero-slope line
    '''
    def line(x,m,b):
        return(m*x+b)
    popt,pcov = sp.optimize.curve_fit(line,np.log(s),lambdas,[0,0.16],lambda_errors)
    m = popt[0]
    dm = np.sqrt(np.diag(pcov))[0]
    print(f"Fitting a line to Lambda^2 for poly_order {i}, we get y=m log(s)+b with:\nm +/- dm = {m:.3f} +/- {dm:.3f}")

In [ ]:
for i in q_values: 
    results, lambdas, lambda_errors, a_s = util.get_res(dfs[i],suppress_print=True)
    lambda_comp( lambdas,lambda_errors,list(results),i)


For each order $q$, the slope $m$ is statistically consistent with zero.

# CONCLUSIONS

In this work we have followed the paper Critical Scale Invariance in a Healthy Human Heart Rate by Kiyono and reproduced the results obtained by the authors. Since the normalized PDF can be effectively superimposed and the $\lambda^2$ parameter from the Castaing fit is constant at different scales within accuracy, we successfully verified the robust scale invariance hypothesis on interbeat time series of healty hearts. 

We have used the dataset Fantasia from [PhysioNet.org](https://physionet.org/) and we have analyzed only one heart beat time series. To improve the work, one could analyze different subjects from different datasets and check how the results changes.

# Appendix : Fantasia subset

PhysioNet
Share
About
Explore 

Search PhysioNet
Log in
Fantasia Database 1.0.0
File: <base>/subset/index.shtml (4,287 bytes) 
<!--#set var="TITLE" value="Fantasia Database Subset"-->
<!--#set var="USELOCALCSS" value="1"-->
<!--#include virtual="/head.shtml"-->


<div class="notice">
<p>This &ldquo;mini-collection&rdquo; of human heart rate data was constructed as a teaching
resource for an intensive course (&ldquo;The Modern Science of Human Aging&rdquo;,
conducted at MIT in October, 1999 under the auspices of <a
href="http://necsi.edu/">NECSI</a>).  As such, this specific collection is
not intended for basic research or publications, for which the <a
href="../">complete database</a>, also available here, may be better suited.
It may be useful, however, in other classroom or tutorial settings, and for
self-guided explorations into the world of biologic complexity.</p>
</div>

<p>
This collection consists of 10 heart beat time series: 5 young subjects
(Y1.txt, Y2.txt, etc) and 5 elderly subjects (O1.txt, O2.txt, etc) from
the <a href="../">Fantasia Database</a>.
You may download <a href="heartbeat.tar">heartbeat.tar</a>&nbsp;(<!--#fsize file="heartbeat.tar" -->),&nbsp;a UNIX tar archive
of this entire mini-collection, also available in gzip-compressed form as
<a href="heartbeat.tar.gz">heartbeat.tar.gz</a>&nbsp;(<!--#fsize file="heartbeat.tar.gz" -->).  If you prefer,
you may download individual recordings:

<div class="edbtable">
<table  style="width: 60%;">
<tr><th >Old</th>
<th >Young</th></tr>
<tr><td><a href="O1.txt">O1.txt</a>
(<a href="o1.hea">o1.hea</a>, <a href="o1.qrs">o1.qrs</a>)</td>
<td><a href="Y1.txt">Y1.txt</a>
(<a href="y1.hea">y1.hea</a>, <a href="y1.qrs">y1.qrs</a>)</td>
</tr>

<tr><td><a href="O2.txt">O2.txt</a>
(<a href="o2.hea">o2.hea</a>, <a href="o2.qrs">o2.qrs</a>)</td>
<td><a href="Y2.txt">Y2.txt</a>
(<a href="y2.hea">y2.hea</a>, <a href="y2.qrs">y2.qrs</a>)</td>
</tr>

<tr><td><a href="O3.txt">O3.txt</a>
(<a href="o3.hea">o3.hea</a>, <a href="o3.qrs">o3.qrs</a>)</td>
<td><a href="Y3.txt">Y3.txt</a>
(<a href="y3.hea">y3.hea</a>, <a href="y3.qrs">y3.qrs</a>)</td>
</tr>

<tr><td><a href="O4.txt">O4.txt</a>
(<a href="o4.hea">o4.hea</a>, <a href="o4.qrs">o4.qrs</a>)</td>
<td><a href="Y4.txt">Y4.txt</a>
(<a href="y4.hea">y4.hea</a>, <a href="y4.qrs">y4.qrs</a>)</td>
</tr>

<tr><td><a href="O5.txt">O5.txt</a>
(<a href="o5.hea">o5.hea</a>, <a href="o5.qrs">o5.qrs</a>)</td>
<td><a href="Y5.txt">Y5.txt</a>
(<a href="y5.hea">y5.hea</a>, <a href="y5.qrs">y5.qrs</a>)</td>
</td></tr>
</table>
</div>

<p>
Records <tt>O1</tt> - <tt>O5</tt> and <tt>Y1</tt> - <tt>Y5</tt> of this subset
correspond to records <tt>f1o01</tt> - <tt>f1o05</tt> and <tt>f1y01</tt> -
<tt>f1y05</tt> of the Fantasia Database, which also includes the ECG and
respiration signals for these recordings.</p>

<p>
Each <tt>.txt</tt> file contains one column of data, consisting of interbeat
intervals (in seconds).  The <tt>.hea</tt> (text header) and <tt>.qrs</tt>
(binary annotation) files contain the same data as in the <tt>.txt</tt> files,
written in the standard formats used for other PhysioBank databases.  The
<tt>.tar</tt> and <tt>.tar.gz</tt> archives include only the <tt>.txt</tt>
files.  The length of each recording is approximately 2 hours.</p>

<p>
Five young (21 - 34 years old) and five elderly (68 - 81 years old)
rigorously-screened healthy subjects underwent 120 minutes of
continuous supine resting while continuous electrocardiographic (ECG)
signals were collected.</p>

<p>
All subjects remained in a resting state in sinus rhythm while
watching the movie
<a href="http://imdb.com/Title?0032455" target="other"><i>Fantasia</i></a> (Disney, 1940) to help maintain wakefulness. The
continuous ECG was digitized at 250 Hz. Each heartbeat was annotated
using an automated arrhythmia detection algorithm, and each beat
annotation was verified by visual inspection. The R-R interval
(interbeat interval) time series for each subject was then computed.</p>

<h2>References</h2>
<p>
These 10 time series form a subset of the <a href="../">complete database</a>
described in:</p>

<div class="reference">
Iyengar N, Peng C-K, Morin R, Goldberger AL, Lipsitz LA. 
Age-related alterations in the fractal scaling of cardiac
interbeat interval dynamics. <i>Am J Physiol</i> 1996;<b>271</b>:1078-1084.
</div> <!-- end reference -->

<!--#include virtual="/dir-footer.shtml"-->
PhysioNet
Maintained by the MIT Laboratory for Computational Physiology

Supported by the National Institute of Biomedical Imaging and Bioengineering (NIBIB), National Heart Lung and Blood Institute (NHLBI), and NIH Office of the Director under NIH grant numbers U24EB037545 and R01EB030362

# Bibliography

<a id="ref1"></a>
[1] Peng, C-K., et al. **"Long-range anticorrelations and non-Gaussian behavior of the heartbeat."** Physical review letters 70.9 (1993): 1343.

<a id="ref2"></a>
[2] P. C. Ivanov et al., Nature (London) 399, 461 (1999).

<a id="ref3"></a>
[3] [Wikipedia page : Heart Rate](https://en.wikipedia.org/wiki/Heart_rate)

<a id="ref4"></a>
[4] Kiyono, **Ken, et al. "Critical scale invariance in a healthy human heart rate."** Physical Review Letters 93.17 (2004): 178103.

<a id="ref4"></a>
[5] Lin, D. C., and R. L. Hughson. **"Modeling heart rate variability in healthy humans: a turbulence analogy."** Physical Review Letters 86.8 (2001): 1650.
